In [0]:

from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
    output_df = input_df.withColumn('ingestion_date', current_timestamp())
    return output_df

In [0]:
def merge_delta_lake(input_df, db_name, table_name, merge_condition, partition_column=None):

    from delta.tables import DeltaTable

    full_table_name = f"{db_name}.{table_name}"

    # 🔍 Validar si la tabla existe (forma confiable)
    tables_df = spark.sql(f"SHOW TABLES IN {db_name}")
    table_exists = tables_df.filter(f"tableName = '{table_name}'").count() > 0

    if table_exists:

        delta_table = DeltaTable.forName(spark, full_table_name)

        (delta_table.alias("tgt")
            .merge(
                input_df.alias("src"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:
        writer = (input_df.write
            .mode("overwrite")
            .format("delta")
        )

        if partition_column is not None:
            writer = writer.partitionBy(partition_column)

        writer.saveAsTable(full_table_name)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import LongType

def convert_fastest_lap_to_ms(time_str):

    if time_str is None:
        return None

    try:

        minutes, seconds = time_str.split(":")
        seconds, milliseconds = seconds.split(".")

        total_ms = (
            int(minutes) * 60 * 1000 +
            int(seconds) * 1000 +
            int(milliseconds)
        )

        return total_ms

    except:
        return None

convert_fastest_lap_udf = udf(
    convert_fastest_lap_to_ms,
    LongType()
)